# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```

In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata
metadata = dataset.metadata
print(f"Dataset: {metadata.name}\n\nDescription: {metadata.description}\n\nPublished: {metadata.datePublished}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

The FAIR² dataset uses Croissant entities referenced by `@id`. Let's explore the available record sets and their schema.

In [ ]:
# List all record sets in the dataset, referencing by @id
record_sets = []
for rs in dataset.record_sets:
    print(f"Record Set Name: {rs.name} | @id: {rs.id}")
    record_sets.append(rs.id)
    print("Fields:")
    for field in rs.fields:
        print(f"  - Field Name: {field.name} | @id: {field.id} | DataType: {getattr(field, 'data_type', 'N/A')}")
    print("---")

# For demonstration, print first few records from the first record set (if available)
if record_sets:
    rs_id = record_sets[0]
    for i, x in enumerate(dataset.records(record_set=rs_id)):
        print(x)
        if i > 2:
            break

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis. All entities are referenced by their `@id` fields.

In [ ]:
dataframes = {}
# Loop over all record sets discovered above
for rs_id in record_sets:
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)
    print("-----")
    print(f"DataFrame for Record Set @id: {rs_id}")
    print(f"Columns: {dataframes[rs_id].columns.tolist()}")
    print(dataframes[rs_id].head())

# Select one record set for further analysis. We'll choose the first for demonstration.
main_record_set_id = record_sets[0] if record_sets else None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering, normalization, grouping, and outlier detection.

We will use field `@id`s for all operations.

In [ ]:
# Identify a numeric field
if main_record_set_id:
    df = dataframes[main_record_set_id]
    # Dynamically attempt to find a numeric column (e.g. Age, IntervalBetweenDiagnosis, etc) via field ids
    numeric_field_id = None
    group_field_id = None
    # Inspect columns
    for col in df.columns:
        # Simple heuristic: Look for fields named containing 'Age' or 'Interval' or numeric dtype
        if 'Age' in col or 'Interval' in col or df[col].dtype in ['int64', 'float64']:
            numeric_field_id = col
            break
    # For grouping, try anatomical location or MSI status
    for col in df.columns:
        if 'AnatomicalLocation' in col or 'MSIStatus' in col or 'Sex' in col:
            group_field_id = col
            break
    
    print(f"Using numeric_field_id: {numeric_field_id}, group_field_id: {group_field_id}")
    
    # If a numeric field exists, filter by threshold
    threshold = 10
    if numeric_field_id and numeric_field_id in df.columns:
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())
        
        # Normalize numeric field
        col_norm = f"{numeric_field_id}_normalized"
        filtered_df[col_norm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, col_norm]].head())
        
        # Group by group_field if available
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"Grouped by {group_field_id} (mean {numeric_field_id}):")
            print(grouped_df.head())

## 5. Visualization
Visualize distributions and relationships between fields.

We visualize the numeric field distribution and a group comparison using matplotlib.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id:
    df = dataframes[main_record_set_id]
    if numeric_field_id and numeric_field_id in df.columns:
        plt.figure(figsize=(8,4))
        sns.histplot(df[numeric_field_id].dropna(), bins=10, kde=True)
        plt.title(f"Distribution of {numeric_field_id}")
        plt.xlabel(numeric_field_id)
        plt.ylabel("Count")
        plt.show()
    if group_field_id and numeric_field_id and group_field_id in df.columns:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()

## 6. Conclusion
Summarize key findings and insights from the FAIR² dataset exploration.

- We loaded the dataset and explored the available record sets and fields using their Croissant `@id` identifiers.
- Data was extracted and processed dynamically, with numeric and categorical fields identified for EDA.
- Filtering and normalization demonstrated how to prepare clinical and molecular data for further analysis.
- Visualizations highlighted data distributions and relationships between clinical attributes.

This notebook illustrates how to use `mlcroissant` for organized, reproducible exploration of FAIR datasets referenced exclusively with `@id` fields.